# Introduction
In this notebook, various ways of plotting 3D information will be explored. The following will be covered:
-  Plot depth maps in 3D axes
- Try out Open3D
- Add Pose Points to 3D plots

Three images are goinf to be used for this example, one of a crashed car(car1.jpg), the UtahTeapot(0.3.jpg) and the ModedCube(33.jjpg). The images are located within the testImages directory.

# Plot depth maps in 3D axes

Using zoe-depth, depths maps of images will be first computed then plotted using Matplotlib.

The plot will be in 3 dimensions.

In [1]:
# Import Libraries
import torch
from zoedepth.models.builder import build_model
from zoedepth.utils.config import get_config
from zoedepth.utils.misc import colorize
import matplotlib.pyplot as plt
import cv2
import numpy as np
from matplotlib.colors import Normalize
import matplotlib.gridspec as gridspec
import open3d as o3d

%matplotlib notebook

# Load the ZoeD_N pretrained Model
conf = get_config("zoedepth", "infer")
zoe = build_model(conf)

# Define Helper Functions

def load_image(img_name, img_dir="testImages/"):
    """
    Loads in an image from the file system

    Args:
    - img_name (string): File name of the image.
    - img_dir (string): The relative directory in which the image is contained.

    Returns:
    - numpy.ndarray: The image as a 2D numpy array
    """
    return cv2.cvtColor(cv2.imread(img_dir+img_name), cv2.COLOR_BGR2RGB)

def load_image_gray(img_name, img_dir="testImages/"):
    """
    Loads in an image from the file system

    Args:
    - img_name (string): File name of the image.
    - img_dir (string): The relative directory in which the image is contained.

    Returns:
    - numpy.ndarray: The image as a 2D numpy array
    """
    return cv2.imread(img_dir+img_name, cv2.IMREAD_GRAYSCALE)

def compute_depth(image, dev="cpu"):
    """
    Loads in an image from the file system

    Args:
    - img_name (string): File name of the image.
    - img_dir (string): The relative directory in which the image is contained.

    Returns:
    - numpy.ndarray: The image as a 2D numpy array
    """
    DEVICE = "cuda" if dev == "cuda" and torch.cuda.is_available() else "cpu"
    zoe.to(DEVICE)
    depth_numpy = zoe.infer_pil(image)  # as numpy
    zoe.to('cpu')
    torch.cuda.empty_cache()  # Clear unused cached memory
    return depth_numpy

def show_image(image, label, colorbar=True):
    """
    Display an image with an optional colorbar.

    Parameters:
    image : array-like
        The image data. This can be any array-like object that is interpretable by `imshow`.
    label : str
        The title label for the image. This text will be displayed above the image.
    colorbar : bool, optional
        A flag to indicate whether a colorbar should be displayed alongside the image.
        If True (default), a colorbar is displayed. If False, no colorbar is shown.

    """
    _, ax = plt.subplots(layout="constrained")
    imgPlot = ax.imshow(image)
    ax.set_title(label)
    if colorbar: plt.colorbar( imgPlot, ax=ax )

img_names = [ "car1.jpg", "33.jpg", "0.3.jpg" ] # list of images to be used
img_labels = ["Car", "Moded Cube", "Utah Teapot"]
depths = dict()

# Load the ZoeD_N pretrained Model
conf = get_config("zoedepth", "infer")
model_zoe_n = build_model(conf)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
img_size [384, 512]


Using cache found in C:\Users\Kibzyzii/.cache\torch\hub\intel-isl_MiDaS_master
E:\im3dr\lib\site-packages\torch\functional.py:513: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\TensorShape.cpp:3610.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Params passed to Resize transform:
	width:  512
	height:  384
	resize_target:  True
	keep_aspect_ratio:  True
	ensure_multiple_of:  32
	resize_method:  minimal
Using pretrained resource url::https://github.com/isl-org/ZoeDepth/releases/download/v1.0/ZoeD_M12_N.pt
Loaded successfully
img_size [384, 512]


Using cache found in C:\Users\Kibzyzii/.cache\torch\hub\intel-isl_MiDaS_master


Params passed to Resize transform:
	width:  512
	height:  384
	resize_target:  True
	keep_aspect_ratio:  True
	ensure_multiple_of:  32
	resize_method:  minimal
Using pretrained resource url::https://github.com/isl-org/ZoeDepth/releases/download/v1.0/ZoeD_M12_N.pt
Loaded successfully


In [ ]:
def depth_map_surface(img_name, label):
    image = load_image(img_name)
    depth_map = compute_depth(image)
    x = np.arange(depth_map.shape[1])
    y = np.arange(depth_map.shape[0])
    X,Y = np.meshgrid(x, y)
    
    # Create a figure with two subplots: one 2D and one 3D
    fig = plt.figure(figsize=(9,9), layout="constrained")

    # Create a gridspec with 2 rows and 2 columns
    gs = gridspec.GridSpec(2, 2, width_ratios=[1, 1], height_ratios=[1,2])


    axImg = fig.add_subplot(gs[0])
    axImg.imshow(image)
    axImg.set_title("2D Image")
    
    axDepth = fig.add_subplot(gs[1])
    axDepth.imshow(colorize(depth_map))
    axDepth.set_title("Depth Map")
    
    ax3d = fig.add_subplot(gs[2:4], projection='3d')
    # Plot the depth map as a 3D surface
    surface = ax3d.plot_surface(X, Y, depth_map, cmap='viridis') 
    ax3d.set_title(label)

    # Set axis labels
    ax3d.set_xlabel('X')
    ax3d.set_ylabel('Y')
    ax3d.set_zlabel('Depth')
    ax3d.set_title("Depths Surface")
    ax3d.view_init(elev=-90, azim=-90, roll=None, vertical_axis='z')
    
    fig.colorbar(surface, ax=ax3d, orientation='vertical', pad=0.1, label='Depth')

    plt.show()
    
img = "33.jpg"
depth_map_surface("33.jpg", "Moded Cube")

# Open3D

[Open3D](open3d.org) is an open-source library that supports rapid development of software that deals with 3D data, providing a set of data structures and algorithms for working with 3D data.The backend is highly optimized and is set up for parallelization.

In [2]:
color_img = load_image("33.jpg")
depth_img = compute_depth(color_img, "cuda")

## Creating Point Clouds from RGBD

In [4]:
# Import the Open3D library
import open3d as o3d

_scale = 1000
color_raw = o3d.geometry.Image(color_img)
depth_raw =  o3d.geometry.Image(depth_img*_scale)

# Create RGBD 
rgbd_image = o3d.geometry.RGBDImage.create_from_color_and_depth(
    color_raw, 
    depth_raw
)

# Camera Focal length
focal_l = 6
f_xy = focal_l *  color_img.shape[1]/6.287
cx, cy = ( color_img.shape[1]/2, color_img.shape[0]/2 )


intrinsics = o3d.camera.PinholeCameraIntrinsic( o3d.camera.PinholeCameraIntrinsicParameters.PrimeSenseDefault)
print("GOT HERE")
# intrinsics = o3d.camera.PinholeCameraIntrinsic(color_img.shape[1], color_img.shape[0], f_xy, f_xy, cx, cy)

# Create Point Cloud
pcd = o3d.geometry.PointCloud.create_from_rgbd_image(
    rgbd_image,
    intrinsics
)

print(f"Points : {len(pcd.points)}")

GOT HERE
Points : 382249


### Modifying Views 

The o3d.visualization.draw_geometries() takes takes the following arguements which modify the perspective of the point cloud

**front**: This argument specifies the "front" direction of the camera. In other words, it defines the direction the camera is facing. It's a vector that points directly out of the camera's lens. The values you provide here determine which direction is considered the front or forward direction in the scene.

**lookat**: This argument sets the point in space that the camera is looking at. It's the focal point of the camera. When you set this parameter, you are effectively positioning the camera so that the lookat point is at the center of the camera's field of view. It's like when you point a camera at a specific object to make sure it's in the shot.

**up**: This argument defines the "up" direction in the scene relative to the camera's perspective. It's a vector that points upwards from the camera's point of view. Setting this vector helps to orient the camera by specifying which direction should be considered up in the visualization.

```
front = [0,0,-1]  # Camera is facing along the negative Z-axis
lookat = [0,0,0]  # Camera is looking at the origin
up = [0,1,0]      # The positive Y-axis is up
```

In [ ]:
# Visualize the Point Cloud
o3d.visualization.draw_geometries([pcd],zoom=.1,
                                  front=[0, 0, -1],
                                  lookat=[0, 0, 0],
                                  up=[0, -1, 0])

`estimate_normals` computes the normal for every point. The function finds adjacent points and calculates the principal axis of the adjacent points using covariance analysis.

The function takes an instance of KDTreeSearchParamHybrid class as an argument. The two key arguments `radius` and `max_nn` specifies search radius and maximum nearest neighbor. It has 10cm of search radius, and only considers up to 30 neighbors to save computation time.

In [ ]:
# Voxel Downsampling and Normal Calculation
downpcd = pcd.voxel_down_sample(voxel_size=.1)

downpcd.estimate_normals( search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=1, max_nn=20))
downpcd.orient_normals_towards_camera_location()

o3d.visualization.draw_geometries([downpcd], point_show_normal=True, 
                                  zoom=.1,
                                  front=[0, 0, -1],
                                  lookat=[0, 0, 0],
                                  up=[0, -1, 0])
print(f"Points: {len(downpcd.points)}")

In [ ]:
# Creating Bounding Volumes
bounding_boxa = downpcd.get_axis_aligned_bounding_box()
bounding_boxa.color = (1,0,0)

bounding_boxb = downpcd.get_oriented_bounding_box()
bounding_boxb.color = (0,0,1)


o3d.visualization.draw_geometries([downpcd, bounding_boxa, bounding_boxb],
                                  zoom=.1,
                                  front=[0, 0, -1],
                                  lookat=[0, 0, 0],
                                  up=[0, -1, 0])

## Clusering

Open3D implements DBSCAN [Ester1996] that is a density based clustering algorithm. The algorithm is implemented in cluster_dbscan and requires two parameters: `eps` defines the distance to neighbors in a cluster and `min_points` defines the minimum number of points required to form a cluster. The function returns labels, where the label -1 indicates noise.

In [ ]:
with o3d.utility.VerbosityContextManager(
        o3d.utility.VerbosityLevel.Debug) as cm:
    labels = np.array(
        pcd.cluster_dbscan(eps=0.09, min_points=100, print_progress=True))

max_label = labels.max()
print(f"point cloud has {max_label + 1} clusters")
colors = plt.get_cmap("tab20")(labels / (max_label if max_label > 0 else 1))
colors[labels < 0] = 0
pcd.colors = o3d.utility.Vector3dVector(colors[:, :3])
o3d.visualization.draw_geometries([pcd], zoom=.1,
                                  front=[0, 0, -1],
                                  lookat=[0, 0, 0],
                                  up=[0, -1, 0])

# Using Masked Depth Maps

In [ ]:
# Helper Functions
from rembg import remove, new_session
def create_mask(image):
    '''
        Create a Mask out of an image
    '''
    model_name = "u2net" # sam, u2net, silueta, isnet-general-use
    session = new_session(model_name)
    mask = remove(image, only_mask=True, post_process_mask=True)
    return mask
    
def mask_out(mask, _img):
    '''
        Masks out part of the image
    '''
    to_mask= np.copy(_img) # create a copy of the depth map
    to_mask[mask == 0] = 0
    return to_mask

In [ ]:
# Mask out both Color and Depth images
mask = create_mask(color_img)
masked_col = mask_out(mask, color_img)
masked_dep = mask_out(mask, depth_img)

fig, axs = plt.subplots( 1, 2, figsize=(8,3), layout="constrained" )
for title, ax, img in zip([ "Colour Image", "Depth Image" ], axs, [ masked_col, masked_dep ]):
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")


In [ ]:
_scale = 1000
color_raw_m = o3d.geometry.Image(masked_col)
depth_raw_m =  o3d.geometry.Image(masked_dep*_scale) 

# Create RGBD 
rgbd_image_m = o3d.geometry.RGBDImage.create_from_color_and_depth(
    color_raw_m, 
    depth_raw_m
)

# Camera Focal length
focal_l = 6
f_xy = focal_l *  color_img.shape[1]/6.287
cx, cy = ( color_img.shape[1]/2, color_img.shape[0]/2 )

intrinsics = o3d.camera.PinholeCameraIntrinsic(color_img.shape[1], color_img.shape[0], f_xy, f_xy, cx, cy)

# Create Point Cloud
pcd2 = o3d.geometry.PointCloud.create_from_rgbd_image(
    rgbd_image_m,
    intrinsics
)


pcd2.estimate_normals( search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=.1, max_nn=30))
pcd2.orient_normals_towards_camera_location()

pcd2_p = np.asarray(pcd2.points)
pcd2_n = np.asarray(pcd2.normals)

# Voxel Downsampling and Normal Calculation
downpcd2 = pcd2.voxel_down_sample(voxel_size=.02)

downpcd2.estimate_normals( search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=.1, max_nn=30))
downpcd2.orient_normals_towards_camera_location()

# Creating Bounding Volumes
bba = downpcd2.get_axis_aligned_bounding_box()
bba.color = (1,0,0)

o3d.visualization.draw_geometries([downpcd2, bba], 
                                  point_show_normal=False,
                                  zoom=.2,
                                  front=[0, 0, -1],
                                  lookat=[0, 0, 0],
                                  up=[0, -1, 0] )

downpcd_p = np.asarray(downpcd2.points)
downpcd_n = np.asarray(downpcd2.normals)

print(f"Points: {len(downpcd2.points)} with shape: {downpcd_p.shape}")
print(f"Dimensions: { bba.get_extent() } ")

In [ ]:
def pixel_to_3d_point(u, v, depth, fx, fy, cx, cy):
    """
    Converts a pixel coordinate to a 3D point using depth and camera intrinsics.

    Args:
    u, v (int): Pixel coordinates.
    depth (float): Depth value at the pixel.
    fx, fy (float): Focal lengths of the camera in pixel units.
    cx, cy (float): Principal point coordinates.

    Returns:
    (float, float, float): The 3D coordinates of the point.
    """
    Z = depth
    X = (u - cx) * Z / fx
    Y = (v - cy) * Z / fy
    return (X, Y, Z)

_, (ax1, ax2) = plt.subplots(1,2, layout="constrained", figsize=(9,4))
ax1.imshow(masked_col)
ax1.axis("off")
ax1.set_title("Color Image")

ax2.imshow(masked_dep)
ax2.axis("off")
ax2.set_title("Color Image")

point1 = (685, 30)
point2 = (685, 730)
p1 = pixel_to_3d_point(point1[0], point1[1],
                depth_img[point1[1], point1[0]],
                  f_xy, f_xy, cx, cy )

p2 = pixel_to_3d_point(point2[0], point2[1],
                depth_img[point2[1], point2[0]],
                  f_xy, f_xy, cx, cy )

print( f"Closest Point: {np.min( depth_img )}" )
print( p2[1]- p1[1] )

# Observations

From the Col_Light Folder, there is an image RL2.jpg. This image was used to get a reference for then depth map accuracies. The real object has a bounding box of ( 0.626, .7, 0.595) meters. The object was placed at the center of the scene but with a vertical displacemet of 0.5 meters(0, 0.5, 0) meters. The camera was placed 1.25 meters in along the global z-axis of the scene. Ths would put it approximately 0.95m away from the closest point of the object to the camera(1.25 - 0.595/2)m

The depth map approximates the closest point to be 1.125m.This would make the depth maps estimation ~1.18 times more from the grond truth. Let us call this the estimate factor or in short `est_factor`.

To valuidate the `pixel_to_3d_point()` we can sample the higher and lower points of the target object with respect to the orinal color image and use this value to estimate their corrsponding coordinates thereby getting a sense of the height. The same can be done for width values but for this case we will consider height. We expect this height to be differ by a the `est_factor` as well.

The points considered were: `(685, 30)` and `(685, 730)`. The difference in height was 0.866m. The ground truth height was 0.7m. Going by the height values, the `*est_factor` should be 1.23. Going by the oringinal **est_factor**, the new height should have been 0.826m.


In [ ]:
0.866/.7

In [ ]:
pcd2_p = np.asarray(pcd2.points)
pcd2_p_r = pcd2_p.reshape(pcd2_p.shape[1], pcd2_p.shape[0])

_ax = 0
print(f"Depth Delta: { np.max(masked_dep) - np.min(masked_dep) }")
print(f"After Reshape: { np.max(pcd2_p_r[_ax]) - np.min(pcd2_p_r[_ax]) }")

In [ ]:
np.max(pcd2_p_r[2]) - np.min(pcd2_p_r[2])

# PyVista

[PyVista](https://docs.pyvista.org/version/stable/) is a versatile 3D visualization library in Python, offering an exceptional platform for visualizing point clouds and other 3D data structures from Open3D. It enables a seamless transition of point cloud data from Open3D to an interactive, high-quality visualization environment. The integration process can be straightforward, typically involving data conversion from Open3D formats to PyVista-compatible formats

PyVista excels in its ability to embed interactive 3D plots within Jupyter Notebooks, providing an immersive experience that Open3D's native visualization tools may lack. For enhanced interactivity in Jupyter, PyVista supports various backend frameworks like trame. To fully utilize these capabilities, additional dependencies such as trame and trame-vtk are required. These can be installed using pip:


```
pip install trame trame-vtk
```

Once installed, setting up the Jupyter backend with PyVista can be done as follows:

```
import pyvista as pv
pv.set_jupyter_backend('trame')
```

```python
import open3d as o3d
import pyvista as pv
import numpy as np

# Convert Open3D point cloud to PyVista format
pcd = o3d.geometry.PointCloud()
np_points = np.random.rand(100, 3)  # Example points
pcd.points = o3d.utility.Vector3dVector(np_points)
points = np.asarray(pcd.points)
pv_cloud = pv.PolyData(points)
```

In [ ]:
import pyvista as pv
pv.set_jupyter_backend('trame')

sphere = pv.Sphere()
sphere.plot()


In [ ]:
# Create a PyVista point cloud
pv_cloud = pv.PolyData(pcd2_p)
pv_cloud['normals'] = pcd2_n

plotter = pv.Plotter(notebook=True)
plotter.add_points(pv_cloud, point_size=2, render_points_as_spheres=True)
# plotter.add_arrows( pcd2_p, pcd2_n, mag=0.1, color='blue' )
plotter.view_isometric()
plotter.show_axes()
plotter.show()